# Overfit sanity check

Trains on 10 examples for 30 epochs and evaluates on the same 10.
Pipeline is healthy if `eval_f1` lands near 1.0.

The last cell dumps per-word gold vs predicted labels (first-subword aggregation, same rule as eval) and prints a per-entity seqeval report.

In [ ]:
!git clone https://github.com/Igufu/dto-fine-tune.git
%cd dto-fine-tune

In [ ]:
!pip install -q -r requirements.txt 'accelerate>=1.1.0'

In [ ]:
!python build_datasets.py

In [ ]:
import json
from datasets import Dataset
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)

from tokenizer import tokenizer, tokenize_and_align_labels
from metrics import LABEL_NAMES, compute_metrics_fns

NAME = "deontic"
label_names = LABEL_NAMES[NAME]
id2label = {i: l for i, l in enumerate(label_names)}
label2id = {l: i for i, l in enumerate(label_names)}

with open(f"labeled_data/{NAME}.jsonl") as f:
    rows = [json.loads(l) for l in f]
sample = [r for r in rows if 1 in r["ner_tags"]][:10]
val_raw = sample  # overfit: train and eval are the same set

ds = Dataset.from_list(sample)
ds = ds.map(tokenize_and_align_labels, batched=True, remove_columns=ds.column_names)

model = AutoModelForTokenClassification.from_pretrained(
    "PlanTL-GOB-ES/RoBERTalex", id2label=id2label, label2id=label2id
)
collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

args = TrainingArguments(
    output_dir="models/overfit",
    num_train_epochs=30,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=5e-5,
    weight_decay=0.0,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds,
    eval_dataset=ds,
    data_collator=collator,
    compute_metrics=compute_metrics_fns[NAME],
    processing_class=tokenizer,
)

trainer.train()
print("final eval:", trainer.evaluate())

In [ ]:
import torch
from seqeval.metrics import classification_report

model.eval()
all_gold, all_pred = [], []

for idx, row in enumerate(val_raw):
    enc = tokenizer(row["tokens"], is_split_into_words=True,
                    truncation=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        preds = model(**enc).logits.argmax(-1)[0].tolist()

    word_pred = {}
    for wid, p in zip(enc.word_ids(), preds):
        if wid is not None and wid not in word_pred:
            word_pred[wid] = p

    n = max(word_pred) + 1
    gold = [label_names[g] for g in row["ner_tags"][:n]]
    pred = [label_names[word_pred[i]] for i in range(n)]
    all_gold.append(gold); all_pred.append(pred)

    for w, g, p in zip(row["tokens"], gold, pred):
        if g != p:
            print(f"ex {idx}: {w:<20} gold={g:<8} pred={p}")

print(classification_report(all_gold, all_pred, zero_division=0))